# Lab 4: Mail Protocols — SMTP, MIME, and Email Forensics
**CMSC395 — Computer Networks**  
Estimated time: 3–4 hours | Points: 100

---

Work through each cell in order.

> **Before starting:** confirm Mailpit is running at `http://localhost:8025`  
> **Submission:** use **Kernel → Restart Kernel and Run All Cells** before your final push.  
> Commit `lab4_session.pcap` to your repository — it is a required artifact.

## Setup — Imports and Configuration

In [ ]:
# Setup — run this cell first
import socket
import time
import uuid
import subprocess
import re
from pathlib import Path
from datetime import datetime, timezone

# Do NOT import base64 — you will implement it yourself in Cell 1.1

MAILPIT_HOST  = 'localhost'
MAILPIT_PORT  = 1025
MAILPIT_UI    = 'http://localhost:8025'

SESSION_PCAP  = Path('lab4_session.pcap')
FORENSICS_PCAP = Path('lab4_forensics.pcap')

FROM_ADDR = 'student@yourclass.edu'
TO_ADDR   = 'recipient@example.com'

# Verify Mailpit is reachable
try:
    s = socket.create_connection((MAILPIT_HOST, MAILPIT_PORT), timeout=3)
    banner = s.recv(256).decode().strip()
    s.close()
    print(f'Mailpit reachable: {banner}')
except Exception as e:
    print(f'WARNING: Cannot reach Mailpit on port {MAILPIT_PORT}: {e}')
    print('Start it with: docker run -d -p 1025:1025 -p 8025:8025 axllent/mailpit')

if not FORENSICS_PCAP.exists():
    print(f'WARNING: {FORENSICS_PCAP} not found — re-run nbgitpuller link to fetch it')
else:
    print(f'Forensics capture: {FORENSICS_PCAP} ({FORENSICS_PCAP.stat().st_size:,} bytes)')

print('Setup complete')

---
## Part 1: SMTP Client with MIME Support

Do not import `smtplib`, `email`, or `base64` — all SMTP and MIME work must use raw sockets
and your own implementations.

### Cell 1.1 — BASE64 Implementation

Implement `base64_encode(data: bytes) -> str` without using Python's `base64` module.

Algorithm:
1. Process the input 3 bytes at a time
2. Treat each group of 3 bytes as a 24-bit integer
3. Split into four 6-bit groups
4. Map each 6-bit value to the BASE64 alphabet
5. Pad with `=` if the input is not a multiple of 3 bytes
6. Wrap output at 76 characters per line

The BASE64 alphabet: `ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/`

Verify your implementation against `base64.b64encode` for at least 5 inputs.

In [ ]:
# Cell 1.1 — BASE64 implementation
# Do NOT import base64 — implement from scratch

BASE64_ALPHABET = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/'

def base64_encode(data: bytes, line_wrap: int = 76) -> str:
    """
    Encode bytes to a BASE64 string.
    Lines are wrapped at line_wrap characters (set to 0 to disable wrapping).
    """
    # Your implementation here
    pass


def base64_decode(s: str) -> bytes:
    """
    Decode a BASE64 string to bytes. (Optional — useful for forensics in Part 3)
    """
    # Your implementation here (optional)
    pass


# Verification against standard library
import base64 as _b64_stdlib   # allowed here only for verification

test_cases = [
    b'',                         # empty
    b'A',                        # 1 byte (2 padding chars)
    b'AB',                       # 2 bytes (1 padding char)
    b'ABC',                      # 3 bytes (no padding)
    b'Hello from Lab 4!',        # longer string
    bytes(range(256)),           # all byte values
    b'student@yourclass.edu',    # used in AUTH LOGIN
]

print(f"{'Input (truncated)':<35} {'Match':>6}")
print('-' * 44)
all_pass = True
for tc in test_cases:
    expected = _b64_stdlib.b64encode(tc).decode()
    # Remove line wrapping for comparison (stdlib doesn't wrap by default)
    got = base64_encode(tc, line_wrap=0)
    match = (got == expected)
    if not match:
        all_pass = False
    label = repr(tc[:20])
    print(f"{label:<35} {'OK' if match else 'FAIL':>6}")
    if not match:
        print(f"  Expected: {expected[:60]}")
        print(f"  Got:      {got[:60]}")

print()
print('All tests passed!' if all_pass else 'Some tests FAILED — fix before continuing')

# Clean up stdlib import — don't use it again
del _b64_stdlib


### Cell 1.2 — MIME Message Builder

Implement `build_mime_message(from_addr, to_addr, subject, text_body, html_body, attachments)`.

- `attachments` is a list of `(filename, content_bytes)` tuples
- Use your `base64_encode` for attachment content
- Generate a unique boundary using `uuid.uuid4()`
- Generate a unique `Message-ID` using hostname + timestamp
- All line endings must be `\r\n`
- Return the complete message as a string (headers + blank line + body)

In [ ]:
# Cell 1.2 — MIME message builder

def make_message_id():
    """Generate a unique Message-ID header value."""
    ts = int(time.time())
    unique = uuid.uuid4().hex[:8]
    hostname = socket.getfqdn()
    return f'<{ts}.{unique}@{hostname}>'


def build_mime_message(from_addr, to_addr, subject,
                        text_body, html_body, attachments=None):
    """
    Build a multipart/mixed MIME message.

    Args:
        from_addr:   sender address string
        to_addr:     recipient address string
        subject:     subject line
        text_body:   plain text body
        html_body:   HTML body
        attachments: list of (filename, content_bytes) tuples

    Returns:
        Complete MIME message as a string with \r\n line endings.
    """
    if attachments is None:
        attachments = []

    boundary = f'lab4-{uuid.uuid4().hex[:16]}'
    now = datetime.now(timezone.utc).strftime('%a, %d %b %Y %H:%M:%S %z')
    message_id = make_message_id()

    lines = []

    # Top-level headers
    lines.append(f'Date: {now}')
    lines.append(f'From: {from_addr}')
    lines.append(f'To: {to_addr}')
    lines.append(f'Subject: {subject}')
    lines.append(f'Message-ID: {message_id}')
    lines.append('MIME-Version: 1.0')
    lines.append(f'Content-Type: multipart/mixed; boundary="{boundary}"')
    lines.append('')   # blank line between headers and body

    # Your MIME parts here:
    # 1. text/plain part
    # 2. text/html part
    # 3. attachment parts (loop over attachments)
    # 4. closing boundary

    return '\r\n'.join(lines)


# Build a test message and verify structure
test_attachment = b'This is the attachment content from Lab 4.\nSecond line.\n'

msg = build_mime_message(
    from_addr=FROM_ADDR,
    to_addr=TO_ADDR,
    subject='Lab 4 Test Message',
    text_body='Hello from Lab 4 — plain text part.',
    html_body='<html><body><p>Hello from <b>Lab 4</b> — HTML part.</p></body></html>',
    attachments=[('lab4_data.txt', test_attachment)]
)

print(f'Message length: {len(msg)} bytes')
print(f'Lines: {msg.count(chr(10))}')
print()
print('--- First 30 lines ---')
for line in msg.split('\r\n')[:30]:
    print(repr(line))


### Cell 1.3 — SMTP Client

Implement `SMTPClient` with `connect()`, `send_message()`, and `quit()`.

- `connect()` opens the socket, reads the banner, sends `EHLO`, and parses the capability list
- `send_message(from_addr, to_addr, message_str)` runs the full `MAIL FROM` → `RCPT TO` → `DATA` sequence
- `quit()` sends `QUIT` and closes the socket
- Raise `SMTPError(code, message)` if any response code is not in the expected success codes
- Implement dot-stuffing in the DATA phase

In [ ]:
# Cell 1.3 — SMTP client

class SMTPError(Exception):
    """Raised when the SMTP server returns an unexpected response code."""
    def __init__(self, code, message):
        self.code = code
        self.message = message
        super().__init__(f'SMTP {code}: {message}')


class SMTPClient:
    """
    Minimal SMTP client using raw sockets.
    Does not use smtplib.
    """

    def __init__(self, host, port):
        self.host = host
        self.port = port
        self._sock = None
        self._file = None
        self.capabilities = {}   # populated by connect()

    def _read_response(self):
        """
        Read a (possibly multi-line) SMTP response.
        Returns (code: int, lines: list[str]).
        Multi-line responses use 'XYZ-text' for continuation and 'XYZ text' for the final line.
        """
        lines = []
        while True:
            line = self._file.readline().rstrip('\r\n')
            lines.append(line)
            # Final line has a space after the 3-digit code
            if len(line) < 4 or line[3] == ' ':
                break
        code = int(lines[-1][:3])
        return code, lines

    def _send_command(self, command, expect=None):
        """
        Send a command and read the response.
        If expect is set, raise SMTPError if the code doesn't match.
        Returns (code, lines).
        """
        self._sock.sendall((command + '\r\n').encode())
        code, lines = self._read_response()
        if expect is not None and code != expect:
            raise SMTPError(code, lines[-1])
        return code, lines

    def connect(self):
        """
        Open connection, read banner, send EHLO, parse capabilities.
        """
        self._sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self._sock.connect((self.host, self.port))
        self._file = self._sock.makefile('r', encoding='utf-8')

        # Read server banner
        code, lines = self._read_response()
        if code != 220:
            raise SMTPError(code, lines[-1])
        print(f'Banner: {lines[0]}')

        # Send EHLO and parse capabilities
        hostname = socket.getfqdn()
        code, lines = self._send_command(f'EHLO {hostname}', expect=250)

        # Parse capability lines (skip the first greeting line)
        print('Server capabilities:')
        for line in lines[1:]:
            # Each capability line: '250-CAPNAME optionalparams' or '250 CAPNAME'
            cap_text = line[4:].strip()   # strip '250-' or '250 '
            parts = cap_text.split(None, 1)
            cap_name = parts[0].upper()
            cap_value = parts[1] if len(parts) > 1 else True
            self.capabilities[cap_name] = cap_value
            print(f'  {cap_name}: {cap_value}')

    def send_message(self, from_addr, to_addr, message_str):
        """
        Send a message via MAIL FROM -> RCPT TO -> DATA.
        message_str is the complete MIME message (headers + body).
        """
        # MAIL FROM
        self._send_command(f'MAIL FROM:<{from_addr}>', expect=250)

        # RCPT TO
        self._send_command(f'RCPT TO:<{to_addr}>', expect=250)

        # DATA
        self._send_command('DATA', expect=354)

        # Send message with dot-stuffing
        # Your dot-stuffing implementation here:
        # Any line beginning with '.' must have an extra '.' prepended
        stuffed_lines = []
        for line in message_str.split('\r\n'):
            if line.startswith('.'):
                line = '.' + line
            stuffed_lines.append(line)
        stuffed = '\r\n'.join(stuffed_lines)

        self._sock.sendall((stuffed + '\r\n.\r\n').encode())
        code, lines = self._read_response()
        if code != 250:
            raise SMTPError(code, lines[-1])
        print(f'Message accepted: {lines[-1]}')

    def authenticate(self, username, password):
        """
        AUTH LOGIN — implemented in Cell 2.1.
        Placeholder here so the class is importable before Part 2.
        """
        raise NotImplementedError('Implement in Cell 2.1')

    def quit(self):
        """Send QUIT and close the connection."""
        try:
            self._send_command('QUIT', expect=221)
        finally:
            if self._sock:
                self._sock.close()
                self._sock = None
                self._file = None


# Connection test
print('Testing SMTP connection to Mailpit...')
client = SMTPClient(MAILPIT_HOST, MAILPIT_PORT)
client.connect()
client.quit()
print('Connection test passed')


### Cell 1.4 — Send Message and Verify in Mailpit

Build a complete multipart message and send it. Then describe what you see in the
Mailpit UI at `http://localhost:8025` in the markdown cell below.

Your message must include:
- A plain text body
- An HTML body with at least one formatting tag
- A file attachment (use `test_attachment` from Cell 1.2 or create your own)

In [ ]:
# Cell 1.4 — Send message and verify in Mailpit

# Build the message
attachment_content = b'Lab 4 attachment.\nThis file was BASE64-encoded by our own implementation.\n'

message = build_mime_message(
    from_addr=FROM_ADDR,
    to_addr=TO_ADDR,
    subject='CMSC395 Lab 4 — MIME Test',
    text_body=(
        'Hello from Lab 4!\n'
        'This is the plain text part of a multipart MIME message.\n'
        'The same message also has an HTML part and a file attachment.\n'
    ),
    html_body=(
        '<html><body>'
        '<h1>Lab 4 — HTML Part</h1>'
        '<p>This is the <b>HTML</b> part of a <em>multipart</em> MIME message.</p>'
        '<p>The message was sent using a <code>raw socket</code> SMTP client.</p>'
        '</body></html>'
    ),
    attachments=[
        ('lab4_attachment.txt', attachment_content)
    ]
)

# Send
print(f'Message size: {len(message)} bytes')
client = SMTPClient(MAILPIT_HOST, MAILPIT_PORT)
client.connect()
client.send_message(FROM_ADDR, TO_ADDR, message)
client.quit()
print(f'\nMessage sent. Check Mailpit UI at {MAILPIT_UI}')

# Print the Message-ID so you can find it in Mailpit
for line in message.split('\r\n'):
    if line.startswith('Message-ID:'):
        print(f'Message-ID: {line[12:]}')
        break


### Mailpit Verification

Open `http://localhost:8025` and find your message. Confirm each item below,
then describe what you see. Double-click to edit.

**Plain text part:** *(Describe what you see — does it render correctly?)*

**HTML part:** *(Does the HTML render? Do the formatting tags appear correctly?)*

**Attachment:** *(Is the attachment present? What filename is shown? Did you verify the content?)*

**Message-ID header:** *(Is it present? What does it look like?)*

### Reflection 1.A

Your MIME message uses a boundary string to delimit parts. What would happen if the boundary
string appeared inside one of the message parts? How does the MIME specification say this
should be handled, and how would you prevent it in your implementation?

**Your answer:**

*(Write here)*

### Reflection 1.B

BASE64 encoding expands data by approximately 33%. Send a message with a 1 KB attachment
and capture the session with tshark. How many bytes did the DATA phase transfer?
How does this compare to the raw attachment size? Where does the extra overhead come from?

**Your answer:**

*(Write here — include the byte counts from your capture)*

---
## Part 2: SMTP AUTH and Session Capture

Extend your SMTP client with AUTH LOGIN and capture the full authenticated session.

**Before running Cell 2.2**, restart Mailpit with auth enabled and start tshark:

```bash
# Restart Mailpit with auth
docker stop $(docker ps -q --filter ancestor=axllent/mailpit)
docker run -d -p 1025:1025 -p 8025:8025 \
  -e MP_SMTP_AUTH_ACCEPT_ANY=true axllent/mailpit

# Start capture in another terminal
tshark -i lo -w ~/lab4_session.pcap -f "tcp port 1025" -a duration:30
```

### Cell 2.1 — AUTH LOGIN Extension

Implement `SMTPClient.authenticate(username, password)` as a standalone function here,
then monkey-patch it onto your existing `SMTPClient` class.

The AUTH LOGIN exchange uses your `base64_encode` implementation for credentials.
Remember to use `line_wrap=0` — AUTH responses must not contain newlines.

In [ ]:
# Cell 2.1 — AUTH LOGIN extension

def _authenticate(self, username, password):
    """
    Perform AUTH LOGIN with the server.
    Raises SMTPError if authentication fails.
    """
    # Send AUTH LOGIN
    code, lines = self._send_command('AUTH LOGIN')
    if code != 334:
        raise SMTPError(code, lines[-1])

    # Decode the challenge (should be BASE64('Username:'))
    challenge = lines[-1][4:]   # strip '334 '
    print(f'Challenge 1: {challenge} → {base64_decode(challenge) if base64_decode(b"") is not None else "(decode not implemented)"}')

    # Send BASE64-encoded username
    encoded_user = base64_encode(username.encode(), line_wrap=0)
    # Your code here — send encoded username and handle the response

    # Send BASE64-encoded password
    encoded_pass = base64_encode(password.encode(), line_wrap=0)
    # Your code here — send encoded password and handle the response

    print(f'Authenticated as {username}')


# Patch authenticate onto SMTPClient
SMTPClient.authenticate = _authenticate

# Test authentication against Mailpit with auth enabled
print('Testing AUTH LOGIN...')
auth_client = SMTPClient(MAILPIT_HOST, MAILPIT_PORT)
auth_client.connect()
auth_client.authenticate('testuser', 'testpassword')
auth_client.quit()
print('Authentication test passed')


### Cell 2.2 — Capture and Analysis

Send an authenticated message (tshark should already be running from the terminal).
Then answer the four questions using tshark programmatically.

After the cell runs, copy the capture:
```bash
cp ~/lab4_session.pcap ./lab4_session.pcap
git add lab4_session.pcap
git commit -m "Lab4: add SMTP session capture"
```

In [ ]:
# Cell 2.2 — Send authenticated message and analyze capture

# Send the message
print('Sending authenticated message...')
auth_client = SMTPClient(MAILPIT_HOST, MAILPIT_PORT)
auth_client.connect()
auth_client.authenticate('student', 'cmsc395')

auth_message = build_mime_message(
    from_addr=FROM_ADDR,
    to_addr=TO_ADDR,
    subject='Lab 4 — Authenticated Message',
    text_body='This message was sent with AUTH LOGIN.',
    html_body='<html><body><p>Authenticated SMTP message.</p></body></html>',
    attachments=[('auth_test.txt', b'A' * 1024)]   # 1 KB attachment
)
auth_client.send_message(FROM_ADDR, TO_ADDR, auth_message)
auth_client.quit()
print('Message sent — stop tshark now and copy the capture file')
print()

# Wait for capture file to be copied
import time
time.sleep(2)

if not SESSION_PCAP.exists():
    print(f'Waiting for {SESSION_PCAP} — copy it now and re-run this cell')
else:
    print(f'Capture found: {SESSION_PCAP.stat().st_size:,} bytes')
    print()

    def tshark(args, pcap=SESSION_PCAP):
        r = subprocess.run(['tshark', '-r', str(pcap)] + args,
                           capture_output=True, text=True)
        return r.stdout

    # Q1: Full SMTP conversation in order
    print('=== Q1: Full SMTP conversation ===')
    # Your tshark command here
    # Hint: tshark -r pcap -q -z "follow,tcp,ascii,0"

    # Q2: AUTH LOGIN credentials decoded
    print('\n=== Q2: AUTH LOGIN credentials ===')
    # Extract the AUTH exchange and decode the BASE64 credentials
    # Your code here

    # Q3: DATA phase TCP segments
    print('\n=== Q3: DATA phase segments ===')
    # Your tshark command here

    # Q4: RTT between final '.' and 250 OK
    print('\n=== Q4: DATA terminator RTT ===')
    # Your tshark command here


### Reflection 2.A

Your capture shows the AUTH LOGIN credentials in BASE64. Decode them and confirm they match
what you sent. Why is BASE64 not encryption? What would an attacker on the same network see?

**Your answer:**

*(Write here — include the decoded credentials from your capture)*

### Reflection 2.B

The EHLO response lists the server's capabilities including `SIZE` with a maximum message size.
What happens if you try to send a message larger than this limit?
Write a test that triggers this and show the server's response.

In [ ]:
# Reflection 2.B — Test: send oversized message
# Build a message larger than the server's SIZE limit and attempt to send it
# Show the server's error response

# Your code here


**Your answer:**

*(What error code and message did the server return?)*

---
## Part 3: Email Forensics

Analyze `lab4_forensics.pcap` to find five deliberate anomalies in an SMTP session.
Each finding must be supported by raw evidence from the capture.

### Cell 3.1 — Extract SMTP Conversation

Use `tshark` to extract the raw SMTP conversation from `lab4_forensics.pcap`.
Print every line exchanged between client and server in chronological order,
labelled with direction (C: for client, S: for server).

In [ ]:
# Cell 3.1 — Extract SMTP conversation from forensics capture

if not FORENSICS_PCAP.exists():
    print(f'ERROR: {FORENSICS_PCAP} not found')
else:
    def forensics_tshark(args):
        r = subprocess.run(['tshark', '-r', str(FORENSICS_PCAP)] + args,
                           capture_output=True, text=True)
        return r.stdout

    # Follow the TCP stream to get the full conversation
    print('=== Full SMTP conversation ===')
    # Your tshark command here
    # Hint: tshark -r pcap -q -z "follow,tcp,ascii,0"
    # The output interleaves client and server lines


### Cell 3.2 — Extract and Parse the Message

Extract the DATA payload from the conversation. Parse:
- All message headers into a dict
- Each MIME part (type, content, encoding)
- The attachment content (decode from BASE64 using your implementation or Python's stdlib)

In [ ]:
# Cell 3.2 — Extract and parse the message

# Step 1: Extract the DATA payload
# The DATA phase starts after '354' and ends at the '.' terminator
# Your extraction code here

raw_conversation = ''   # populate from Cell 3.1 output

def extract_data_payload(conversation):
    """
    Extract the message body from an SMTP conversation string.
    Returns everything between '354' response and the final '.' line.
    """
    lines = conversation.split('\n')
    in_data = False
    payload_lines = []

    for line in lines:
        if '354' in line:
            in_data = True
            continue
        if in_data:
            if line.strip() == '.':
                break
            payload_lines.append(line)

    return '\n'.join(payload_lines)


def parse_headers(header_block):
    """
    Parse email headers from a header block string.
    Returns a dict. Multi-line headers (folded) are joined.
    """
    headers = {}
    current_key = None
    current_val = []

    for line in header_block.split('\n'):
        if line.startswith((' ', '\t')) and current_key:
            # Folded header continuation
            current_val.append(line.strip())
        elif ':' in line:
            if current_key:
                headers[current_key] = ' '.join(current_val)
            current_key, _, val = line.partition(':')
            current_key = current_key.strip()
            current_val = [val.strip()]
        elif not line.strip() and current_key:
            break   # blank line = end of headers

    if current_key:
        headers[current_key] = ' '.join(current_val)

    return headers


# Your parsing code here
# Extract DATA, split headers from body, parse headers, identify MIME parts
print('Parsing forensics message...')


### Cell 3.3 — Five Forensic Findings

For each finding, show the raw evidence and your conclusion.  
Write code to extract the evidence, then explain it in the markdown cell below.

In [ ]:
# Cell 3.3 — Forensic analysis code

import base64 as _stdlib_b64   # allowed here for decoding forensics content

# Finding 1: Envelope vs header sender mismatch
print('=== Finding 1: Envelope vs header sender ===')
# Extract MAIL FROM from conversation
# Extract From: header from message
# Compare and report


# Finding 2: Received header chain
print('\n=== Finding 2: Received header chain ===')
# Extract all Received: headers
# Parse MTA hostnames and timestamps
# Display in chronological order


# Finding 3: Attachment magic bytes
print('\n=== Finding 3: Attachment content vs declared type ===')
# Find the attachment MIME part
# Decode from BASE64
# Read first 8 bytes and identify file type
# Compare with Content-Type header


# Finding 4: Message-ID analysis
print('\n=== Finding 4: Message-ID ===')
# Extract and parse the Message-ID header
# Report what it reveals about the sending system


# Finding 5: Timestamp anomaly
print('\n=== Finding 5: Timestamp analysis ===')
# Extract Date: header and all Received: header timestamps
# Parse them and check chronological order
# Report any anomaly


### Forensic Findings Summary

Write one paragraph per finding. Reference specific values from the capture.

---

**Finding 1 — Envelope vs header sender mismatch:**

*(Write here)*

---

**Finding 2 — Received header chain:**

*(Write here — include the MTA chain in chronological order)*

---

**Finding 3 — Mistyped attachment:**

*(Write here — include the magic bytes and the declared vs actual file type)*

---

**Finding 4 — Message-ID analysis:**

*(Write here — what does the Message-ID reveal?)*

---

**Finding 5 — Timestamp anomaly:**

*(Write here — describe the anomaly and what it suggests)*

### Reflection 3.A

The `From:` header and `MAIL FROM:` envelope address are different. Email clients typically
show users only the `From:` header. What kind of attack does this enable?
What email authentication mechanisms exist to detect or prevent it?

**Your answer:**

*(Write here)*

### Reflection 3.B

You found that the attachment's declared `Content-Type` did not match its actual content.
In a real email security context, why is this significant?
What defenses check for this mismatch?

**Your answer:**

*(Write here)*

### Reflection 3.C

The `Received:` header timestamps showed an anomaly. What does this tell you about
the message's history? In a forensic investigation, how would you distinguish a legitimate
clock skew from deliberate header manipulation?

**Your answer:**

*(Write here)*

---
## Submission Checklist

Before your final push, complete this checklist. Replace each `[ ]` with `[x]` when done.

- [ ] Cell 1.1: BASE64 verified against stdlib for all 5+ test cases
- [ ] Cell 1.2: MIME builder produces valid multipart message with 3 parts
- [ ] Cell 1.3: SMTP client connects, sends EHLO, capability list printed
- [ ] Cell 1.4: Message sent, Mailpit verification written
- [ ] Reflections 1.A and 1.B completed
- [ ] Cell 2.1: AUTH LOGIN implemented and tested
- [ ] Cell 2.2: All 4 capture questions answered
- [ ] `lab4_session.pcap` committed to repository
- [ ] Reflections 2.A and 2.B completed (including oversized message test)
- [ ] Cell 3.1: Full SMTP conversation extracted from forensics capture
- [ ] Cell 3.2: Headers and MIME parts parsed
- [ ] Cell 3.3: All 5 findings with raw evidence shown
- [ ] Forensic findings summary written for all 5 findings
- [ ] Reflections 3.A, 3.B, and 3.C completed
- [ ] Notebook runs cleanly with Kernel → Restart Kernel and Run All Cells
- [ ] At least 5 commits with descriptive messages in repository history

Final commit message must be exactly: `Lab4 final submission`

---
*CMSC395 Computer Networks — Lab 4*  
*Push this notebook to your repository with the message: `Lab4 final submission`*